In [1]:
# ============================================================================
# 步骤 1: 导入必要的库
# ============================================================================
import sys
import os
sys.path.append(os.path.dirname(os.getcwd()))  # 添加项目根目录到Python路径
from qiskit import QuantumCircuit
from qibo import Circuit  # 用于执行
from src.cross_framework_interface import optimize_qiskit

# ============================================================================
# 步骤 2: 创建 Qiskit 电路（假设这是用户已有的电路）
# ============================================================================
# 创建一个 Bell 态电路
qc_qiskit = QuantumCircuit(2)
qc_qiskit.h(0)          # Hadamard 门
qc_qiskit.cx(0, 1)      # CNOT 门
qc_qiskit.h(0)          # 冗余的 H 门（将被优化掉）
qc_qiskit.x(1)          # X 门
qc_qiskit.x(1)          # 冗余的 X 门（将被优化掉）

print(f"原始 Qiskit 电路: {len(qc_qiskit)} 个门, 深度 {qc_qiskit.depth()}")

# ============================================================================
# 步骤 3: 使用本项目优化并转换为 Qibo 电路
# ============================================================================
# 方案 A: 使用默认 Qiskit Transpiler 优化（推荐）


原始 Qiskit 电路: 5 个门, 深度 4


In [2]:

optimized_qibo = optimize_qiskit(
    qc_qiskit,
    strategy="qiskit_only",    # 纯 Qiskit 优化
    optimization_level=2       # 优化级别 0-3
)




INFO:cross_framework_optimizer:检测到输入类型: qiskit
INFO:cross_framework_optimizer:应用Qiskit优化级别 2
INFO:qiskit.passmanager.base_tasks:Pass: ContainsInstruction - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: UnitarySynthesis - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: HighLevelSynthesis - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: BasisTranslator - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: ElidePermutations - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: RemoveDiagonalGatesBeforeMeasure - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: RemoveIdentityEquivalent - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: InverseCancellation - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: ContractIdleWiresInControlFlow - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: CommutativeCancellation - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: ConsolidateBlocks - 0.00000 (ms)
INFO:qiskit.passmanager.base_tasks:Pass: Split2QUn

In [3]:
optimized_qibo.draw()

0: ─U3─o─U3─
1: ────X────


In [5]:

# ============================================================================
# 步骤 4: 在 Qibo 中执行优化后的电路
# ============================================================================
# 执行电路
result = optimized_qibo()

print("\n测量结果:")
print(f"量子态: {result}")  # 显示最终的量子态



测量结果:
量子态: (-0.5-0j)|00> + (-0.5+0j)|01> + (-0.5-0j)|10> + (0.5+0j)|11>


In [ ]:

# ============================================================================
# (可选) 步骤 5: 获取详细统计信息
# ============================================================================
from src.cross_framework_interface import optimize_circuit_with_stats

optimized, stats = optimize_circuit_with_stats(
    qc_qiskit,
    strategy="qiskit_only",
    optimization_level=2,
    verbose=False
)

print("\n详细统计:")
print(f"  门减少: {stats['gate_reduction']} ({stats['gate_reduction_percent']:.1f}%)")
print(f"  优化时间: {stats['optimization_time']:.4f}s")
print(f"  转换时间: {stats['conversion_time']:.4f}s")